# add-cxr-to-multimodal-task
---
## Setup

In [ ]:
%%capture

from datetime import datetime
from typing import Any, Dict, List, Optional
import os
import polars as pl

# PyHealth Packages
from pyhealth.datasets import MIMIC4Dataset
from pyhealth.tasks.multimodal_mimic4 import ClinicalNotesMIMIC4, ClinicalNotesICDLabsMIMIC4, ClinicalNotesICDLabsCXRMIMIC4
from pyhealth.tasks.base_task import BaseTask

# Load MIMIC4 Files
# There's probably better ways dealing with this on the cluster, but working locally for now 
# (see: https://github.com/sunlabuiuc/PyHealth/blob/master/examples/mortality_prediction/multimodal_mimic4_minimal.py)

TASK = "ClinicalNotesICDLabsCXRMIMIC4" # The idea here is that we want additive tasks so we can evaluate the value in adding more modalities

PYHEALTH_REPO_ROOT = '/Users/wpang/Desktop/PyHealth'

EHR_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "local_data/local/data/physionet.org/files/mimiciv/2.2")
NOTE_ROOT = os.path.join(PYHEALTH_REPO_ROOT, "local_data/local/data/physionet.org/files/mimic-iv-note/2.2")
CXR_ROOT = os.path.join(PYHEALTH_REPO_ROOT,"local_data/local/data/physionet.org/files/mimic-cxr-jpg/2.0.0")
CACHE_DIR = os.path.join(PYHEALTH_REPO_ROOT,"local_data/local/data/wp/pyhealth_cache")

DEV_MODE = True

if __name__ == "__main__":

    if TASK == "ClinicalNotesMIMIC4": # A bit janky setup at the moment and open to iteration, but conveys the point for now
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )
    
    elif TASK == 'ClinicalNotesICDLabsMIMIC4':
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )

    elif TASK == 'ClinicalNotesICDLabsCXRMIMIC4':
        dataset = MIMIC4Dataset(
                ehr_root=EHR_ROOT,
                note_root=NOTE_ROOT,
                cxr_root=CXR_ROOT,
                ehr_tables=["diagnoses_icd", "procedures_icd", "prescriptions", "labevents"],
                note_tables=["discharge", "radiology"],
                cxr_tables=["metadata", "negbio"],
                cache_dir=CACHE_DIR,
                num_workers=8,
                dev=DEV_MODE
            )

In [ ]:
# negbio_patient_ids = (                                                                                                
#       dataset.global_event_df                                                                                           
#       .filter(pl.col("event_type") == "negbio")                                                                         
#       .select("patient_id")                                                                                             
#       .unique()                                                                                                         
#       .collect(engine="streaming")
#       .to_series()                                                                                                      
#       .to_list()                                                                                                        
#   )  

# OR 
# dataset.unique_patient_ids[2]

ID = "15092875"
print(f"Patient ID is {ID}")

In [ ]:
# Single patient
patient = dataset.get_patient(ID)  

### XRay

In [ ]:
%%capture
negbio_events = patient.get_events(event_type="negbio")
print(len(negbio_events))
print(getattr(negbio_events[0], "pneumonia", None))
print(negbio_events[4])

In [ ]:
%%capture
metadata_events = patient.get_events(event_type="metadata")
print(metadata_events)

### Run Task 

In [ ]:
# Apply multimodal task
task = ClinicalNotesICDLabsCXRMIMIC4() 
samples = task(patient)

In [ ]:
samples